In [ ]:
%%capture
# Normally using pip install unsloth is enough

# Temporarily as of Jan 31st 2025, Colab has some issues with Pytorch
# Using pip install unsloth will take 3 minutes, whilst the below takes <1 minute:
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
!pip install --no-deps cut_cross_entropy unsloth_zoo
!pip install --no-deps unsloth
!pip install boto3

In [ ]:
!pip list

In [ ]:
import os
import torch
import boto3
import pandas as pd
from io import StringIO
from datasets import load_dataset, Dataset

# Adding model libraries to train
from trl import SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import TrainingArguments, DataCollatorForSeq2Seq

In [ ]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
import os
s3client = boto3.client('s3', region_name='us-east-2',
                        aws_access_key_id = os.environ["AWS_ACCESS_KEY_ID"],
                        aws_secret_access_key = os.environ["AWS_SECRET_ACCESS_KEY"])


bucket = os.environ["S3_BUCKET"]
s3Object = "KnowledgeBase/Processed/ConvDataFinalVibhorg"
local_path = "./temp.parquet"

s3client.download_file(bucket, f"{s3Object}.parquet", local_path)
df = pd.read_parquet(local_path)
conv_dataset = Dataset.from_pandas(df)

In [ ]:
def format_chat_template(attribute):
    if attribute["History"] ==  None:
        attribute["History"] = ""

    json_format = [{"role": "system", "content": SYSTEM_PROMPT + attribute["History"]},
                   {"role": "user", "content": attribute["Context"]},
                   {"role": "assistant101", "content": attribute["Response"]}]

    attribute["trainText"] = tokenizer.apply_chat_template(json_format,
                                                      tokenize=False)

    return attribute

final_conversations = conv_dataset.map(format_chat_template, num_proc=2)